# Recurrent Neural Networks (RNNs) for NLP
## Comprehensive Guide

This notebook covers all 6 requested areas.


---
## 1. Mathematical Intuition

### Hidden State Equations
The RNN hidden state is computed at each time step t using:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

Where:
* $h_t$: Hidden state at time $t$ (vector of size `hidden_dim`)
* $x_t$: Input at time $t$ (vector of size `input_dim`)
* $W_{xh}$: Input-to-hidden weight matrix (`hidden_dim` $\times$ `input_dim`)
* $W_{hh}$: Hidden-to-hidden weight matrix (`hidden_dim` $\times$ `hidden_dim`)
* $b_h$: Bias vector (size `hidden_dim`)r


---
## 2. Core Concepts

### Why RNNs for Sequential Data?
1. Parameter Sharing: same weights at each timestep
2. Hidden State: memory mechanism
3. Variable Length: any sequence length

### Architecture Variants:
- Many-to-One: Sentiment analysis
- Many-to-Many: Translation, NER


---
## 3. Applications in NLP

1. **Sentiment Analysis** (Many-to-One)
2. **Machine Translation** (Many-to-Many)
3. **Text Generation** (Many-to-Many)
4. **Named Entity Recognition** (Many-to-Many)
5. **Text Summarization** (Many-to-Many)


---
## 4. Implementation

Complete PyTorch implementation for sentiment analysis.


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter


In [35]:
class Vocabulary:
    def __init__(self, min_freq=1):
        self.word2idx = {"PAD": 0, "UNK": 1}
        self.idx2word = {0: "PAD", 1: "UNK"}
        self.min_freq = min_freq
    
    def build_vocab(self, texts):
        counts = Counter()
        for t in texts:
            counts.update(t.lower().split())
        for w, c in counts.items():
            if c >= self.min_freq and w not in self.word2idx:
                idx = len(self.idx2word)
                self.word2idx[w] = idx
                self.idx2word[idx] = w
    
    def encode(self, text):
        return [self.word2idx.get(w, 1) for w in text.lower().split()]
    
    def __len__(self):
        return len(self.idx2word)


In [36]:
class RNNSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.rnn(embedded)
        return self.fc(hidden.squeeze(0))


In [37]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100):
        self.texts, self.labels, self.vocab, self.max_len = texts, labels, vocab, max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.vocab.encode(self.texts[idx])
        if len(enc) < self.max_len:
            enc += [0] * (self.max_len - len(enc))
        else:
            enc = enc[:self.max_len]
        return torch.tensor(enc), torch.tensor(self.labels[idx])


In [38]:
sample_texts = ["great movie loved it", "terrible waste time", "amazing story", "bad film avoid"]
sample_labels = [1, 0, 1, 0]
vocab = Vocabulary()
vocab.build_vocab(sample_texts)
print(f"Vocab: {len(vocab)}")


Vocab: 14


In [39]:
model = RNNSentiment(len(vocab), 50, 64, 2)
dataset = TextDataset(sample_texts, sample_labels, vocab)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)


RNNSentiment(
  (embedding): Embedding(14, 50, padding_idx=0)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [40]:
for epoch in range(5):
    for texts, labels in loader:
        optimizer.zero_grad()
        out = model(texts)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.6935
Epoch 2, Loss: 0.6932
Epoch 3, Loss: 0.6933
Epoch 4, Loss: 0.6931
Epoch 5, Loss: 0.7115


---
## 4.5: Numerical Walkthrough with Numpy

Let's compute the RNN forward pass step-by-step with concrete numbers to make the mathematics tangible.

### Setup
- Vocabulary: {PAD: 0, good: 1, movie: 2}
- Hidden size: 2 (small for clarity)
- Sequence: ["good", "movie"]
- One-hot encoding for input representation


In [41]:
import numpy as np

# Weight matrices (toy values for clarity)
# Note: For PyTorch compatibility, weight_ih_l0 has shape (hidden_size, input_size)
W_xh = np.array([[0.1, 0.2, 0.5], [0.3, 0.4, 0.6]])  # shape (hidden=2, input=3)
W_hh = np.array([[0.1, 0.0], [0.0, 0.1]])
b_h = np.array([0.1, 0.2])
W_hy = np.array([[1.0, 0.5], [0.5, 1.0]])
b_y = np.array([0.0, 0.0])

vocab_map = {"PAD": 0, "good": 1, "movie": 2}

def one_hot(idx, vocab_size):
    vec = np.zeros(vocab_size)
    if idx > 0:
        vec[idx] = 1.0
    return vec

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

print("Weight matrices:")
print(f"W_xh:\n{W_xh}\n")
print(f"W_hh:\n{W_hh}\n")
print(f"b_h: {b_h}\n")
print(f"W_hy:\n{W_hy}\n")
print(f"b_y: {b_y}")


Weight matrices:
W_xh:
[[0.1 0.2 0.5]
 [0.3 0.4 0.6]]

W_hh:
[[0.1 0. ]
 [0.  0.1]]

b_h: [0.1 0.2]

W_hy:
[[1.  0.5]
 [0.5 1. ]]

b_y: [0. 0.]


In [42]:
h_0 = np.array([0.0, 0.0])
print(f"Initial hidden state h_0: {h_0}\n")

x_1 = one_hot(vocab_map["good"], 3)
print(f"Step 1: Input 'good' -> one-hot: {x_1}")
pre_act1 = W_xh @ x_1 + W_hh @ h_0 + b_h
print(f"   pre-activation = W_xh @ x_1 + W_hh @ h_0 + b_h = {pre_act1}")
h_1 = np.tanh(pre_act1)
print(f"   h_1 = tanh(pre-activation) = {h_1}\n")

x_2 = one_hot(vocab_map["movie"], 3)
print(f"Step 2: Input 'movie' -> one-hot: {x_2}")
pre_act2 = W_xh @ x_2 + W_hh @ h_1 + b_h
print(f"   pre-activation = W_xh @ x_2 + W_hh @ h_1 + b_h = {pre_act2}")
h_2 = np.tanh(pre_act2)
print(f"   h_2 = tanh(pre-activation) = {h_2}\n")


Initial hidden state h_0: [0. 0.]

Step 1: Input 'good' -> one-hot: [0. 1. 0.]
   pre-activation = W_xh @ x_1 + W_hh @ h_0 + b_h = [0.3 0.6]
   h_1 = tanh(pre-activation) = [0.29131261 0.53704957]

Step 2: Input 'movie' -> one-hot: [0. 0. 1.]
   pre-activation = W_xh @ x_2 + W_hh @ h_1 + b_h = [0.62913126 0.85370496]
   h_2 = tanh(pre-activation) = [0.55745373 0.69300007]



In [43]:
y = W_hy @ h_2 + b_y
print(f"Output: y = W_hy @ h_2 + b_y = {y}")
p = softmax(y)
print(f"Probabilities: p = softmax(y) = {p}")
print(f"   Negative prob = {p[0]:.4f}, Positive prob = {p[1]:.4f}")


Output: y = W_hy @ h_2 + b_y = [0.90395377 0.97172694]
Probabilities: p = softmax(y) = [0.48306319 0.51693681]
   Negative prob = 0.4831, Positive prob = 0.5169


In [44]:
# Verify our numpy computation matches PyTorch
# Note: PyTorch uses bias_ih + bias_hh, so we divide b_h to match single-bias formulation
rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True)
rnn.weight_ih_l0 = nn.Parameter(torch.tensor(W_xh, dtype=torch.float32))
rnn.weight_hh_l0 = nn.Parameter(torch.tensor(W_hh, dtype=torch.float32))
rnn.bias_ih_l0 = nn.Parameter(torch.tensor(b_h / 2, dtype=torch.float32))
rnn.bias_hh_l0 = nn.Parameter(torch.tensor(b_h / 2, dtype=torch.float32))

fc = nn.Linear(2, 2)
fc.weight = nn.Parameter(torch.tensor(W_hy, dtype=torch.float32))
fc.bias = nn.Parameter(torch.tensor(b_y, dtype=torch.float32))

x_np1 = one_hot(vocab_map["good"], 3)
x_np2 = one_hot(vocab_map["movie"], 3)
x_torch = torch.tensor(np.array([x_np1, x_np2]), dtype=torch.float32).unsqueeze(0)

output_torch, hidden_torch = rnn(x_torch)
print(f"PyTorch all outputs:\n{output_torch.detach().numpy()}")
print(f"PyTorch final hidden: {hidden_torch.detach().numpy().flatten()}")

y_torch = fc(hidden_torch.squeeze(0))
p_torch = torch.softmax(y_torch, dim=1)
print(f"PyTorch predictions: {p_torch.detach().numpy().flatten()}")


PyTorch all outputs:
[[[0.29131263 0.5370496 ]
  [0.55745375 0.6930001 ]]]
PyTorch final hidden: [0.55745375 0.6930001 ]
PyTorch predictions: [0.48306316 0.5169368 ]


---
## 5. Limitations

### Vanishing Gradient Problem
- Gradients become exponentially small through time
- Early inputs receive near-zero updates
- Solutions: Gradient clipping, LSTM/GRU, residual connections

### Exploding Gradient Problem
- Gradients grow exponentially through time
- Causes unstable training
- Solutions: Gradient clipping (norm threshold), careful initialization

### Long-Term Dependencies
- Standard RNNs struggle with dependencies beyond ~10 timesteps
- Information gets diluted through repeated multiplications
- LSTM and GRU architectures address this with gating mechanisms

### Computational Limitations
- Sequential computation prevents parallelization
- Slow training on long sequences
- Modern alternatives: Transformers (self-attention)


---
## 6. Mini-Project: Sentiment Analysis

### Dataset Suggestions
- IMDb Movie Reviews
- Twitter Sentiment
- Amazon Product Reviews

### Preprocessing Steps
1. Load dataset
2. Clean text: remove HTML, URLs, special characters
3. Lowercase and tokenize
4. Build vocabulary with word frequency threshold
5. Convert text to sequences of indices
6. Pad/truncate sequences to fixed length
7. Split into train/validation/test sets (80/10/10)

### Model Architecture
- Embedding Layer: 100-300 dimensions
- RNN Layer: 128-256 hidden units
- Dropout: 0.3-0.5
- Fully Connected: output layer with softmax

### Evaluation Metrics
- Accuracy
- Precision/Recall/F1-score
- Confusion Matrix
- ROC-AUC (for binary classification)

### Expected Improvements
- Replace RNN with LSTM/GRU
- Add bidirectional RNN
- Use pre-trained embeddings (GloVe, Word2Vec)
- Implement attention mechanism
